In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_exposure_adjustment_stepup_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema =(
        #         "ExposureAdjustmentStepUpId string,"
        #         "SourceSystem string,"
        #         "PortfolioKey string,"
        #         "PortfolioName string,"
        #         "IngestionTimestamp timestamp,"
        #         "BatchId string,"
        #         "ExposureAdjustmentStepUpKey string,"
        #         "ExposureKey string,"
        #         "ExposureUnit int,"
        #         "ExpsosureType int,"
        #         "ExposureSN int,"
        #         "ChangeSN int,"
        #         "CalculationDate timestamp,"
        #         "CalculationIndex string,"
        #         "FixInterestCode int,"
        #         "FixInterest string,"
        #         "FixStartDate timestamp,"
        #         "FixEndDate timestamp,"
        #         "FixScaleDate timestamp,"
        #         "FixAutoRenewalFlag string,"
        #         "FixCancelFlag string,"
        #         "FloatInterestCode int,"
        #         "FloatInterest string,"
        #         "PenaltyInterestCode int,"
        #         "PenaltyInterest string,"
        #         "ControllingInterestCode int,"
        #         "ControllingInterest string,"
        #         "Transactionuser string,"
        #         "Comments string,"
        #         "TransactionDatetime timestamp,"
        #         "IsActiveFlag string,"
        #         "ProcessedStatus string,"
        #         "ProcessedDatetime timestamp,"
        #         "ProcessedError string,"
        #         "UnfreezeDate date,"
        #         "FreezeFlag string,"
        #         "InstanceNo string,"
        #         "ValidDate timestamp,"
        #         "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExposureAdjustmentStepUp"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "profits",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ============================== loan_change_order ====================
        loan_change_order_extra_filters = None
        if loan_change_order_extra_filters is not None:
            loan_change_order_extra_filters.append(date_filter)
            loan_change_order_final_filters = " AND ".join(
                loan_change_order_extra_filters
            )
        else:
            loan_change_order_final_filters = date_filter
        loan_change_order_selected_fields = [
            "ACC_UNIT",
            "ACC_TYPE",
            "ACC_SN",
            "CHANGE_SN",
            "CALCULATION_DATE",
            "CALCULATION_IND",
            "FIX_INTEREST",
            "FIX_START_DATE",
            "FIX_END_DATE",
            "FIX_SCALE_DATE",
            "FIX_AUTO_RENEWAL",
            "FIX_CANCEL_DATE",
            "FLOAT_INTEREST",
            "PENALTY_INTEREST",
            "CONTR_INTEREST",
            "TRX_USER",
            "COMMENTS",
            "TMSTAMP",
            "ENTRY_STATUS",
            "PROCESSED_STATUS",
            "PROCESSED_TMSTAMP",
            "PROCESSED_ERROR",
            "UN_FREEZE_DATE",
            "FREEZE_FLAG",
            "INSTANCE_NO",
            "VALIDITY_DATE",
            "BusinessDate",
        ]

        loan_change_order = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_change_order",
            loan_change_order_final_filters,
            loan_change_order_selected_fields,
        )

        # ============================== loan_details ====================
        loan_details_extra_filters = None
        if loan_details_extra_filters is not None:
            loan_details_extra_filters.append(date_filter)
            loan_details_final_filters = " AND ".join(loan_details_extra_filters)
        else:
            loan_details_final_filters = date_filter
        loan_details_selected_fields = [
            "prft_acc_unit",
            "prft_acc_type",
            "prft_acc_sn",
            "account_number",
        ]

        loan_details = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_details",
            loan_details_final_filters,
            loan_details_selected_fields,
        )

        # ============================== loan_interest ====================
        loan_interest_extra_filters = None
        if loan_interest_extra_filters is not None:
            loan_interest_extra_filters.append(date_filter)
            loan_interest_final_filters = " AND ".join(loan_interest_extra_filters)
        else:
            loan_interest_final_filters = date_filter
        loan_interest_selected_fields = [
            "interest_description",
            "interest_code",
        ]

        loan_interest = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_interest",
            loan_interest_final_filters,
            loan_interest_selected_fields,
        )

        loan_interest = loan_interest.distinct()

        loan_interest_fix = loan_interest.withColumnRenamed(
            "interest_code", "interest_code_fix"
        ).withColumnRenamed("interest_description", "interest_description_fix")

        loan_interest_float = loan_interest.withColumnRenamed(
            "interest_code", "interest_code_float"
        ).withColumnRenamed("interest_description", "interest_description_float")

        loan_interest_penalty = loan_interest.withColumnRenamed(
            "interest_code", "interest_code_penalty"
        ).withColumnRenamed("interest_description", "interest_description_penalty")

        loan_interest_contr = loan_interest.withColumnRenamed(
            "interest_code", "interest_code_contr"
        ).withColumnRenamed("interest_description", "interest_description_contr")


        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # =================== Join Conditions ==============================

        LoanChangeOrder_LoanDetails_join_condition = (
            (loan_change_order["ACC_UNIT"] == loan_details["prft_acc_unit"])
            & (loan_change_order["ACC_TYPE"] == loan_details["prft_acc_type"])
            & (loan_change_order["ACC_SN"] == loan_details["prft_acc_sn"])
        )

        LoanChangeOrder_Portfolio_join_condition = (
            loan_change_order["ACC_UNIT"] == portfolio["unit_code_profits"]
        )

        loan_change_order_loan_interest_fix_join_condition = (
            loan_change_order["FIX_INTEREST"] == loan_interest_fix["interest_code_fix"]
        )

        loan_change_order_loan_interest_float_join_condition = (
            loan_change_order["FLOAT_INTEREST"]
            == loan_interest_float["interest_code_float"]
        )

        loan_change_order_loan_interest_penalty_join_condition = (
            loan_change_order["PENALTY_INTEREST"]
            == loan_interest_penalty["interest_code_penalty"]
        )

        loan_change_order_loan_interest_contr_join_condition = (
            loan_change_order["CONTR_INTEREST"]
            == loan_interest_contr["interest_code_contr"]
        )

        # ========================= Apply the Joins ================

        Joined_df = (
            loan_change_order.join(
                loan_details,
                LoanChangeOrder_LoanDetails_join_condition,
                "inner",
            )
            .join(
                loan_interest_fix,
                loan_change_order_loan_interest_fix_join_condition,
                "left",
            )
            .join(
                loan_interest_float,
                loan_change_order_loan_interest_float_join_condition,
                "left",
            )
            .join(
                loan_interest_penalty,
                loan_change_order_loan_interest_penalty_join_condition,
                "left",
            )
            .join(
                loan_interest_contr,
                loan_change_order_loan_interest_contr_join_condition,
                "left",
            )
            .join(
                portfolio,
                LoanChangeOrder_Portfolio_join_condition,
                "left",
            )
        )

        # ================= Set the Keys for Hashkey================

        ExposureAdjustmentStepUpId_primary_key_fields = [
            loan_change_order.ACC_UNIT,
            loan_change_order.ACC_TYPE,
            loan_change_order.ACC_SN,
            loan_change_order.CHANGE_SN,
            lit("profits"),
        ]

        ExposureAdjustmentStepUpKey_primary_key_fields = [
            loan_change_order.ACC_UNIT,
            loan_change_order.ACC_TYPE,
            loan_change_order.ACC_SN,
            loan_change_order.CHANGE_SN,
        ]

        # =============== Creation of Hashkey ===================
        target_table_profits = Joined_df.withColumn(
            "ExposureAdjustmentStepUpId",
            sha2(
                concat_ws(";", *ExposureAdjustmentStepUpId_primary_key_fields),
                256,
            ),
        ).withColumn(
            "ExposureAdjustmentStepUpKey",
            concat_ws(";", *ExposureAdjustmentStepUpKey_primary_key_fields),
        )

        # =========== Selection of Final Columns ================

        result_df_profits = target_table_profits.select(
            target_table_profits.ExposureAdjustmentStepUpId.cast("string").alias(
                "ExposureAdjustmentStepUpId"
            ),
            lit("profits").cast("string").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            target_table_profits.ExposureAdjustmentStepUpKey.cast("string").alias(
                "ExposureAdjustmentStepUpKey"
            ),
            loan_details.account_number.cast("string").alias("ExposureKey"),
            loan_change_order.ACC_UNIT.cast("int").alias("ExposureUnit"),
            loan_change_order.ACC_TYPE.cast("int").alias("ExpsosureType"),
            loan_change_order.ACC_SN.cast("int").alias("ExposureSN"),
            loan_change_order.CHANGE_SN.cast("int").alias("ChangeSN"),
            loan_change_order.CALCULATION_DATE.cast("timestamp").alias(
                "CalculationDate"
            ),
            loan_change_order.CALCULATION_IND.cast("string").alias("CalculationIndex"),
            loan_change_order.FIX_INTEREST.cast("int").alias("FixInterestCode"),
            loan_interest_fix.interest_description_fix.alias("FixInterest"),
            loan_change_order.FIX_START_DATE.cast("timestamp").alias("FixStartDate"),
            loan_change_order.FIX_END_DATE.cast("timestamp").alias("FixEndDate"),
            loan_change_order.FIX_SCALE_DATE.cast("timestamp").alias("FixScaleDate"),
            loan_change_order.FIX_AUTO_RENEWAL.cast("string").alias(
                "FixAutoRenewalFlag"
            ),
            loan_change_order.FIX_CANCEL_DATE.cast("string").alias("FixCancelFlag"),
            loan_change_order.FLOAT_INTEREST.cast("int").alias("FloatInterestCode"),
            loan_interest_float.interest_description_float.alias("FloatInterest"),
            loan_change_order.PENALTY_INTEREST.cast("int").alias("PenaltyInterestCode"),
            loan_interest_penalty.interest_description_penalty.alias("PenaltyInterest"),
            loan_change_order.CONTR_INTEREST.cast("int").alias(
                "ControllingInterestCode"
            ),
            loan_interest_contr.interest_description_contr.alias("ControllingInterest"),
            loan_change_order.TRX_USER.cast("string").alias("Transactionuser"),
            loan_change_order.COMMENTS.cast("string").alias("Comments"),
            loan_change_order.TMSTAMP.cast("timestamp").alias("TransactionDatetime"),
            loan_change_order.ENTRY_STATUS.cast("string").alias("IsActiveFlag"),
            loan_change_order.PROCESSED_STATUS.cast("string").alias("ProcessedStatus"),
            loan_change_order.PROCESSED_TMSTAMP.cast("timestamp").alias(
                "ProcessedDatetime"
            ),
            loan_change_order.PROCESSED_ERROR.cast("string").alias("ProcessedError"),
            loan_change_order.UN_FREEZE_DATE.cast("date").alias("UnfreezeDate"),
            loan_change_order.FREEZE_FLAG.cast("string").alias("FreezeFlag"),
            loan_change_order.INSTANCE_NO.cast("string").alias("InstanceNo"),
            loan_change_order.VALIDITY_DATE.cast("timestamp").alias("ValidDate"),
            loan_change_order.BusinessDate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df_profits.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_exposure_adjustment_stepup_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)